In [ ]:
# from google.colab import files
# files.upload()

In [ ]:
# !pip install lpips
# !pip install SimpleITK
# !pip install piq
# !pip install torch_fidelity
# !pip install torchmetrics

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# !cp -r "/content/drive/MyDrive/Colab Notebooks/2D_nii_Dataset_skip" /content/

In [ ]:
# !ls /content/

In [ ]:
#del sys.modules['mymodel']
import importlib
import Helper_Functions
import Models   #Hu_2019  #Yang_2025   #Mehraban_2025   #DRUNet
import os
from collections import defaultdict
import json
_=importlib.reload(Helper_Functions)
_=importlib.reload(Models)

In [ ]:
import torch
from torch import nn
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.model_selection import KFold
_=torch.manual_seed(111)  # Random Generator Seed

In [ ]:
device=Helper_Functions.Device()
generators_List = [ Models.UNet,Models.ResUNet,Models.DenseUNet, Models.DRUNet]
batchs_List=[16,8,4,2]
# path=fr"/content/drive/MyDrive/Colab Notebooks/"
# root_dir=fr"/content/drive/MyDrive/Colab Notebooks/2D_nii_Dataset_skip"
path=r"C:\Users\fhabi\Desktop\PhD\DataSet"
root_dir=r"C:\Users\fhabi\Desktop\PhD\DataSet\2D_nii_Dataset_skip"
dataset = Helper_Functions.Load_nii_Files_From_Dataset2(root_dir)   

In [ ]:
results = defaultdict(dict)

metrics_names = ["MSE", "MAE", "PSNR", "SSIM", "MSSSIM", "FID", "LPIPS"]
kf = KFold(n_splits=2,shuffle=True,random_state=42)
for model_class in generators_List:
    model_name = model_class.__name__
    for batch in batchs_List:
        hyperParameters=Helper_Functions.HyperParameters(batch_size=batch,num_epochs=1)
        fold_Train_results = [] 
        fold_Test_results = []   # store 5 folds
        for fold, (train_idx, test_idx) in enumerate(kf.split(dataset)):
            print(f"\n ========== Model:{model_name}, Batch Size:{batch}, Fold:{fold+1} ==========")

            generator = model_class().to(device)
            discriminator = Models.Discriminator().to(device)

            train_dataset = torch.utils.data.Subset(dataset, train_idx)
            test_dataset  = torch.utils.data.Subset(dataset, test_idx)
            train_loader,test_loader=Helper_Functions.Train_Test_DataLoader(train_dataset, test_dataset,batch)
            gan=Helper_Functions.GAN(generator,discriminator,hyperParameters,train_loader,test_loader,path)
            gan.Train(fold+1)
            avg_Train_metrics=gan.CalculateExperimentsResults(train_loader)
            avg_Test_metrics=gan.CalculateExperimentsResults(test_loader)
            
            fold_Train_results.append(avg_Train_metrics)
            fold_Test_results.append(avg_Test_metrics)

        mean_Train_metrics=Helper_Functions.CalculateMeanMetrics(fold_Train_results)
        mean__Test_metrics=Helper_Functions.CalculateMeanMetrics(fold_Test_results)

        gan.SaveResults(mean_Train_metrics,"Train")
        gan.SaveResults(mean__Test_metrics,"Test")
        # --------------------------
        # Save for plotting
        # --------------------------
        for metric in metrics_names:

            results.setdefault(metric, {})
            results[metric].setdefault(model_name, [])

            results[metric][model_name].append(mean__Test_metrics[metric])


file_path = os.path.join(path, "Results.json")

with open(file_path, "w") as f:
    json.dump(results, f, indent=4)
        

In [ ]:
with open(path+"\Results.txt", "w") as f:
    json.dump(results, f, indent=4)